In [6]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os.path as op
import os
import glob
import h5py
import pickle

In [3]:
from functions_avalanches import avalanches_detection
import matplotlib.pyplot as plt
# from pipeline_functions_ATM import MetricsOfInterest
from functions_avalanches import compute_ATM
from scipy.stats import zscore
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import nbimporter
# from functions_avalanches import my_compute_ATM
from functions_avalanches import compute_ATM
from scipy.signal import butter, sosfiltfilt

In [7]:
# generate a list of matrices. Each matrix represents the data of a subject, with shape (n_channels, n_times)
data_all=np.random.rand(39, 64, 1000)  # 39 subjects, 64 channels, 1000 time points

In [11]:
# compute the minimum duration across all subjects to ensure that we can compare the signals on the same number of time points
# in this way the differetn duration of the signals will not affect the computation of the number of neuronal avalanches and the ATM features, which are sensitive to the duration of the signal
duration_signal = {}
for subject in range(39):
    duration_signal[subject] = data_all[subject].shape[0]
min_duration = min(duration_signal.values())

In [ ]:
# Containers: per subject -> result
ATM_by_subject = {}
avalanche_counts_by_subject = {}
flexibility_by_subject = {}
threshold = 2.5  # Z-score threshold for binarization
metricsATM_by_subject = {}

for subject in range(39):
    # 2) reshape is data is (n_channels, n_trials*n_times)
    data = data_all[subject]
    # option 1: consider all the channels
    # option 2: consider only a subset of channels (e.g., frontal channels) X= data[frontal_channels, :].T
    X = data[:,0:80].T
    print(f"X shape (channels x samples): {X.shape}")
    # Binarize (per region along time axis)
    binarized_start = (np.abs(X) > threshold).astype(np.int8)
  
    # computeATM wants (n_regions, n_timepoints, n_subjects)  
    binarized = np.expand_dims(binarized_start[:, :], axis=2)
    print(f"Binarized matrix shape: {binarized.shape},sub {subject}")
    # to not symmterice the ATM, we need to compute the ATM for each subject using compute_ATM_no_simm function
    # the function compute_ATM has an option to consider only avalanches with a minimum duration of 2 time points, which is the minimum duration to compute a transition matrix
    min_timepoints_for_transition_matrix = 2 # default value is 2
    _, ATM_sc, avalanche_counts_sc,flexibility_sc = compute_ATM(binarized[:, :min_duration, :],)
    ATM_by_subject[subject] = ATM_sc
    avalanche_counts_by_subject[subject] = avalanche_counts_sc
    flexibility_by_subject[subject] = flexibility_sc
    
    # starting from the ATM computed with compute_ATM, compute the metrics of interest
    # option 1: compute the metrics of interest considering all the channels involved in ATM matrix
    # option 2: compute the metrics of interest considering only a subsets of channels
    metrics_sc = MetricsOfInterest(ATM_sc[0])
    metricsATM_by_subject[subject] = metrics_sc
        

X shape (channels x samples): (80, 64)
Binarized matrix shape: (80, 64, 1),sub 0


IndexError: index 0 is out of bounds for axis 0 with size 0